In [210]:
import pandas as pd
import numpy as np
import os 
from sklearn.impute import KNNImputer
from sklearn.neighbors import KNeighborsClassifier

In [211]:
DETAIL_PATH = os.path.join("raw", "details_data.csv")
GENERAL_PATH = os.path.join("raw", "general_data.csv")
details = pd.read_csv(DETAIL_PATH)
general = pd.read_csv(GENERAL_PATH)

In [212]:
details = details.drop('floor', axis = 'columns')

In [213]:
merged = general.merge(details, left_on="slug", right_on="slug")

In [214]:
merged = merged.drop_duplicates().reset_index(drop = True)

In [215]:
merged.head(2)

,slug,title,price,square footage,rooms,floor,district,market,build_year,house_type,heating,condidtion,extras,urls
0,2-pokojowe-mieszkanie-45m2-balkon-bezposrednio...,2-pokojowe mieszkanie 45m2 + balkon Bezpośrednio,702000.0,45.00,TWO,FIRST,Podgórze Duchackie,primary,2027.0,block,urban,to_completion,"balcony,garage,lift",https://ireland.apollo.olxcdn.com/v1/files/eyJ...
1,4-pokojowe-mieszkanie-75m2-ogrodek-ID4uk3x,4-pokojowe mieszkanie 75m2 + ogródek,1080432.0,75.03,FOUR,GROUND,Prądnik Czerwony,primary,2026.0,NaN,NaN,to_completion,"garage,garden,lift",https://ireland.apollo.olxcdn.com/v1/files/eyJ...


In [216]:
merged = merged.dropna(subset=['price', 'square footage', 'rooms', 'floor'])
room_map = {'ONE': 1, 'TWO': 2, 'THREE': 3, 'FOUR': 4, 'FIVE':5, 'SIX':6,'SEVEN': 7, 'EIGHT':8, 'NINE':9, 'TEN':10, 'MORE':11}
floor_map = {'FIRST' : 1, 'SECOND' : 2, 'FIFTH': 5,'EIGHTH':8,'THIRD':3, 'FOURTH':4,'CELLAR':-1,'SIXTH' :6, 'SEVENTH':7,'TENTH':10,'NINTH' : 9,'GARRET':11, 'ABOVE_TENTH':11,'GROUND':0}
merged = pd.get_dummies(merged, columns=['district'], dtype = int)
merged['rooms'] = merged['rooms'].map(room_map)
merged['floor'] = merged['floor'].map(floor_map)

In [217]:
# Implement KNN to the missing year ;) 
imputer = KNNImputer(n_neighbors=5)
district_col = [col for col in merged.columns if col.startswith('district_')]
cols_to_model = ['price', 'square footage', 'floor', 'rooms', 'build_year'] + district_col


In [218]:
filled_data = imputer.fit_transform(merged[cols_to_model])

In [219]:
merged[cols_to_model] = np.round(filled_data)

In [220]:
# KNeighbours Classifer 
def fullfill_gaps(df, col_to_predict, cols_to_model):
    normal_data = df[df[col_to_predict].notna()]
    missing_data = df[df[col_to_predict].isna()]
    if len(missing_data) == 0: return df    

    X_train = normal_data[cols_to_model]
    y_train = normal_data[col_to_predict]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_train, y_train)

    missing_X = missing_data[cols_to_model]
    prediction = clf.predict(missing_X)

    df.loc[df[col_to_predict].isna(), col_to_predict] = prediction
    return df 

In [221]:
merged = fullfill_gaps(merged, "heating", cols_to_model)
merged = fullfill_gaps(merged, "house_type", cols_to_model)
merged['extras'] = merged['extras'].fillna('')

In [222]:
merged = merged.dropna(subset=['urls', 'condidtion'])

In [223]:
#balcony, garage, lift, garden, basement, air_condiditoning - i ll be using this parameters

In [224]:
for extra in ["balcony", "garage", "lift", "garden", "basement", "air_conditioning"]:
    merged[f"extras_{extra}"] = merged["extras"].str.contains(f"{extra}", case = False).astype(int)


In [225]:
merged["extras_balcony"].sum()

np.int64(3533)

In [226]:
merged = pd.get_dummies(merged, columns=['house_type', 'heating', 'condidtion'], dtype = int)

In [ ]:
data_for_model = merged.drop(columns=['slug', 'title', 'urls', 'market', 'extras'])

In [ ]:
data_for_model.head(2) # data set - ready to use!

,price,square footage,rooms,floor,build_year,extras,district_Bieńczyce,district_Bieżanów-Prokocim,district_Bronowice,district_Czyżyny,...,house_type_ribbon,house_type_tenement,heating_boiler_room,heating_electrical,heating_gas,heating_other,heating_urban,condidtion_ready_to_use,condidtion_to_completion,condidtion_to_renovation
0,702000.0,45.0,2.0,1.0,2027.0,"balcony,garage,lift",0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,1,0,1,0
1,1080432.0,75.0,4.0,0.0,2026.0,"garage,garden,lift",0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,1,0,1,0
